In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np


In [ ]:
DATABASE_PATH = "../basketball_reference.db"

conn = sqlite3.connect(DATABASE_PATH)
cursor = conn.cursor()

def run_query(query):
    return pd.read_sql_query(query, conn)

In [ ]:
# Question 1


jordan_trophy = """
SELECT p.name, p.height, 'Jordan Trophy' AS label
FROM players p
JOIN award_season aws ON p.player_id = aws.player_id
JOIN awards a ON aws.award_id = a.award_id
WHERE a.name LIKE '%Michael Jordan Trophy%' 
  AND aws.season_id BETWEEN 2020 AND 2024;
"""

top_50_players = """
SELECT p.name, p.height, 'Top 50' AS label
FROM players p
JOIN player_stats ps ON p.player_id = ps.player_id
WHERE ps.season_id BETWEEN 2020 AND 2024
ORDER BY ps.win_shares DESC
LIMIT 50;
"""

In [ ]:
# Question 2


champions_stats = """
SELECT p.name, p.height, ps.experience, 'Champion Team' AS label, ps.season_id
FROM players p
JOIN player_stats ps ON p.player_id = ps.player_id
JOIN seasons s ON ps.season_id = s.season_id
WHERE ps.team_id = s.champion_id 
  AND ps.season_id IN (2023, 2024);
"""


top_15_stats = """
WITH RankedPlayers AS (
    SELECT p.name, p.height, ps.experience, 'Top 15' AS label, ps.season_id,
           ROW_NUMBER() OVER (PARTITION BY ps.season_id ORDER BY ps.win_shares DESC) as rank
    FROM players p
    JOIN player_stats ps ON p.player_id = ps.player_id
    WHERE ps.season_id IN (2023, 2024)
)
SELECT name, height, experience, label, season_id
FROM RankedPlayers
WHERE rank <= 15;
"""

In [ ]:
#Question3

point_guards_jordan = """
SELECT p.player_id, p.name, COUNT(aws.id) AS trophy_count
FROM players p
JOIN player_position pp ON p.player_id = pp.player_id
JOIN award_season aws ON p.player_id = aws.player_id
JOIN awards a ON aws.award_id = a.award_id
WHERE (pp.position = LIKE '%Point Guard%')
  AND a.name LIKE '%Michael Jordan Trophy%'
  AND aws.season_id BETWEEN 2020 AND 2024
GROUP BY p.player_id, p.name
ORDER BY trophy_count DESC, p.name ASC
LIMIT 3;
"""

In [ ]:
#hypothesis test

hypothesis_1 = """
WITH RankedSeason AS (
    SELECT p.player_id, p.height, p.weight, ps.season_id,
           (CAST(p.height AS REAL) / p.weight) AS agility,
           ROW_NUMBER() OVER (PARTITION BY ps.season_id ORDER BY ps.points DESC) as rank
    FROM players p
    JOIN player_stats ps ON p.player_id = ps.player_id
)
SELECT 
    season_id, 
    agility,
   CASE 
        WHEN season_id IN (2021, 2022) THEN 'Past_Group'   --  2020-2021 , 2021-2022
        WHEN season_id IN (2023, 2024) THEN 'Recent_Group' --  2022-2023 , 2023-2024
    END AS period
FROM RankedSeason
WHERE rank <= 20 
  AND season_id IN (2021, 2022, 2023, 2024);
"""

hypothesis_2 = """
WITH DataWithAge AS (
    SELECT 
        ps.season_id,
        ps.experience,
        (ps.season_id - CAST(STRFTIME('%Y', p.birthdate) AS INTEGER)) AS age
    FROM players p
    JOIN player_stats ps ON p.player_id = ps.player_id
    JOIN seasons s ON ps.season_id = s.season_id
    WHERE ps.team_id = s.champion_id 
      AND ps.season_id IN (2021, 2022, 2023, 2024)
)
SELECT 
    season_id,
    experience,
    age,
    (CAST(experience AS REAL) / age) AS innate_ability,
    CASE 
        WHEN season_id IN (2021, 2022) THEN 'Past_Group'  --  2020-2021 , 2021-2022
        WHEN season_id IN (2023, 2024) THEN 'Recent_Group'  --  2022-2023 , 2023-2024
    END AS period
FROM DataWithAge;
"""